# 🎲 Day 20: Double Q-Learning & Maximization Bias
### 100 Days of Machine Learning | Syed Muhammad Qamran
**Topic:** Overestimation in Q-Learning & Two-Estimator Decoupling  
**Benchmark:** Sutton & Barto Example 6.7 (The Maximization Bias MDP)

---

### The Mathematical Problem:
Standard Q-learning uses $\max_a Q(s', a)$ for both selecting and evaluating the best action:
$$\mathbb{E}[\max(X_1, X_2, \dots)] \ge \max(\mathbb{E}[X_1], \mathbb{E}[X_2], \dots)$$
This leads to severe positive maximization bias in stochastic environments.

### The Double Q-Learning Solution:
Maintain two independent Q-tables ($Q_1$ and $Q_2$). Decouple action selection from action evaluation:
$$a^* = \arg\max_a Q_1(s', a)$$
$$\text{Target} = R + \gamma Q_2(s', a^*)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline


In [ ]:
# Maximization Bias MDP Environment (Sutton & Barto Ex. 6.7)
# State A (0): Left -> State B (1), Right -> Terminal (reward 0)
# State B (1): Has 10 actions, each returning Normal(mean=-0.1, std=1.0) reward

def run_maximization_bias_simulation(runs=1000, episodes=300):
    left_q = np.zeros(episodes)
    left_dq = np.zeros(episodes)

    for r in range(runs):
        # Standard Q
        q_a = np.zeros(2)
        q_b = np.zeros(10)
        # Double Q
        q1_a, q2_a = np.zeros(2), np.zeros(2)
        q1_b, q2_b = np.zeros(10), np.zeros(10)

        for ep in range(episodes):
            # 1. Standard Q Episode
            # Action at A (0: Left, 1: Right)
            a = np.random.randint(2) if np.random.rand() < 0.1 else np.argmax(q_a)
            if a == 0:
                left_q[ep] += 1
                b_action = np.random.randint(10) if np.random.rand() < 0.1 else np.argmax(q_b)
                r_b = np.random.normal(-0.1, 1.0)
                q_b[b_action] += 0.1 * (r_b - q_b[b_action])
                q_a[0] += 0.1 * (0.0 + np.max(q_b) - q_a[0])
            else:
                q_a[1] += 0.1 * (0.0 - q_a[1])

            # 2. Double Q Episode
            avg_a = (q1_a + q2_a) / 2.0
            a_dq = np.random.randint(2) if np.random.rand() < 0.1 else np.argmax(avg_a)
            if a_dq == 0:
                left_dq[ep] += 1
                avg_b = (q1_b + q2_b) / 2.0
                b_act = np.random.randint(10) if np.random.rand() < 0.1 else np.argmax(avg_b)
                r_b = np.random.normal(-0.1, 1.0)
                if np.random.rand() < 0.5:
                    best_b = np.argmax(q1_b)
                    q1_a[0] += 0.1 * (0.0 + q2_b[best_b] - q1_a[0])
                    q1_b[b_act] += 0.1 * (r_b - q1_b[b_act])
                else:
                    best_b = np.argmax(q2_b)
                    q2_a[0] += 0.1 * (0.0 + q1_b[best_b] - q2_a[0])
                    q2_b[b_act] += 0.1 * (r_b - q2_b[b_act])
            else:
                if np.random.rand() < 0.5:
                    q1_a[1] += 0.1 * (0.0 - q1_a[1])
                else:
                    q2_a[1] += 0.1 * (0.0 - q2_a[1])

    return (left_q / runs) * 100, (left_dq / runs) * 100

pct_q, pct_dq = run_maximization_bias_simulation(runs=500, episodes=300)

# Replicating Sutton & Barto Figure 6.7
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(pct_q, label="Standard Q-Learning (Severe Overestimation)", color="#EF4444", lw=2)
ax.plot(pct_dq, label="Double Q-Learning (Unbiased)", color="#10B981", lw=2)
ax.axhline(5, color="black", linestyle="--", label="Optimal Behavior (~5%)")
ax.set_title("Replication of Sutton & Barto Fig 6.7: Action Left Percentage from State A", fontsize=12, fontweight='bold')
ax.set_xlabel("Episodes")
ax.set_ylabel("% Actions Left")
ax.set_ylim(0, 100)
ax.legend(fontsize=11)
plt.show()
